# **(Data Cleaning)**

## Objectives

* "Here, we will check all the images and make sure we dont have any data represented as text or any corrupted files"
* We will also plot distribution between training, validation, and testing sets

## Inputs

* inputs/skin_cancer_dataset/benign
* inputs/skin_cancer_dataset/malignant

## Outputs

* No files will be placed in outputs, however, we will be forming the train, validation, and test folders under inputs/skin_cancer_datasets

## Additional Comments

No additional comments

---

# Change working directory

* We are assuming you will store the notebooks in a subfolder, therefore when running the notebook in the editor, you will need to change the working directory
  
We need to change the working directory from its current folder to its parent folder
* We access the current directory with os.getcwd()


In [1]:
import os
current_dir = os.getcwd()
current_dir

'/workspaces/skin-cancer-detector/jupyter_notebooks'

We want to make the parent of the current directory the new current directory
* os.path.dirname() gets the parent directory
* os.chir() defines the new current directory

In [2]:
os.chdir(os.path.dirname(current_dir))
print("You set a new current directory")

You set a new current directory


Confirm new current directory

In [3]:
cuddent_dir = os.getcwd()
current_dir

'/workspaces/skin-cancer-detector/jupyter_notebooks'

---

# Data Cleaning

## Excluding Files

Here, we will check all the images and make sure we dont have any data represented as text

In [4]:
from pathlib import Path
def clean_skin_cancer_dataset(my_data_dir):
    allowed_extensions = {'.png', '.jpeg', '.jpg'}
    class_names = ['benign', 'malignant']
    results = {}

    for class_name in class_names:
        class_dir = Path(my_data_dir) / class_name

        if not class_dir.exists():
            results[class_name] = {'status': 'folder not found', 'kept': 0, 'removed': 0}
            continue

        kept = 0
        removed = 0

        for file_path in class_dir.rglob('*'):
            if file_path.is_file():
                if file_path.suffix.lower() in allowed_extensions:
                    kept += 1
                else:
                    file_path.unlink()
                    removed += 1

        results[class_name] = {'status': 'cleaned', 'kept': kept, 'removed': removed}

    return results

cleaning_results = clean_skin_cancer_dataset('inputs/skin_cancer_dataset')
cleaning_results


{'benign': {'status': 'cleaned', 'kept': 1800, 'removed': 0},
 'malignant': {'status': 'cleaned', 'kept': 1497, 'removed': 0}}

The code above makes sure that the data without the .png, .jpeg, or .jpg extensions are not permitted

## Excluding corrupt files

Here, we mwillbe removin image sthat are corrupted or cannot be opened

In [5]:
from pathlib import Path
from PIL import Image, UnidentifiedImageError


def remove_corrupt_images(data_dir):
    class_names = ['benign', 'malignant']
    results = {}

    for class_name in class_names:
        class_dir = Path(data_dir) / class_name
        removed_files = []

        if not class_dir.exists():
            results[class_name] = {
                'status': 'folder not found',
                'removed': removed_files,
            }
            continue

        for file_path in class_dir.iterdir():
            if not file_path.is_file():
                continue

            try:
                with Image.open(file_path) as image:
                    image.verify()
            except (UnidentifiedImageError, OSError, SyntaxError):
                file_path.unlink()
                removed_files.append(str(file_path))

        results[class_name] = {
            'status': 'cleaned',
            'removed': removed_files,
        }

    return results


corrupt_file_results = remove_corrupt_images('inputs/skin_cancer_dataset')
corrupt_file_results

{'benign': {'status': 'cleaned', 'removed': []},
 'malignant': {'status': 'cleaned', 'removed': []}}

---

# Splitting Train, Validation, and Test Sets

Here, we wseparate all the images into three parts. Training, validation, and test sets.

In [6]:
def split_train_validation_test_set_images(my_data_dir, train_set_ratio, validation_set_ratio, test_set_ratio):
    class_names = ['benign', 'malignant']
    results = {}

    for class_name in class_names:
        class_dir = Path(my_data_dir) / class_name

        if not class_dir.exists():
            results[class_name] = {'status': 'folder not found', 'train': 0, 'validation': 0, 'test': 0}
            continue

        image_files = list(class_dir.glob('*'))
        total_images = len(image_files)

        train_count = int(total_images * train_set_ratio)
        validation_count = int(total_images * validation_set_ratio)
        test_count = total_images - train_count - validation_count

        train_dir = Path(my_data_dir) / 'train' / class_name
        validation_dir = Path(my_data_dir) / 'validation' / class_name
        test_dir = Path(my_data_dir) / 'test' / class_name

        train_dir.mkdir(parents=True, exist_ok=True)
        validation_dir.mkdir(parents=True, exist_ok=True)
        test_dir.mkdir(parents=True, exist_ok=True)

        for i, file_path in enumerate(image_files):
            if i < train_count:
                destination = train_dir / file_path.name
            elif i < train_count + validation_count:
                destination = validation_dir / file_path.name
            else:
                destination = test_dir / file_path.name

            file_path.rename(destination)

        results[class_name] = {'status': 'split completed', 'train': train_count, 'validation': validation_count, 'test': test_count}

    return results

We separate the data into the training with with 70% of data, the validation with 10% of data, and lastly, the testing set with the last 20% of data

In [7]:
split_train_validation_test_set_images(my_data_dir = 'inputs/skin_cancer_dataset',
                                       train_set_ratio=0.7,
                                       validation_set_ratio=0.1,
                                       test_set_ratio=0.2,
                                        )

{'benign': {'status': 'split completed',
  'train': 1260,
  'validation': 180,
  'test': 360},
 'malignant': {'status': 'split completed',
  'train': 1047,
  'validation': 149,
  'test': 301}}

Since we have taken out al the images from both the malignant and benign folders in inputs/skin_cancer_dataset, we can remove them

In [8]:
import os

os.rmdir("inputs/skin_cancer_dataset/benign")
os.rmdir("inputs/skin_cancer_dataset/malignant")

NOTE:

Something I found interesting is that if you keep the directory open on the left side for any files that are being removed or edited, it will not show the chages, however, the computer will recognize the changes. I am not sure if this is my computer or a known bug

---

# Conclusions and Next Steps

We have concluded that the data is clean(does not need to delete any files) and we have split the data into the train, validation, and test sets.

Next, we will move onto data visualization for the dashboard necessities

Also, since we only obtained the inputs folder, there is no need to put outputs since we still have not obtained any outputs for what will be displayed in the dashboard

Lastly, do the following commands in the terminal to save the files

git add .

git commit -m "message you want to add"

git push